In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

I0000 00:00:1792515629.262654   42660 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792515629.267068   42660 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792515629.695016   42660 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792515631.990411   42660 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [4]:
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
model=load_model('simplernn_model.keras')
model.summary()

E0000 00:00:1792515635.180802   42660 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/kartik/.local/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(store)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,626,052 (10.02 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,313,027 (5.01 MB)

In [6]:
model.get_weights()

[array([[ 0.00417958, -0.0112025 ,  0.02191656, ...,  0.03000289,
         -0.04529039, -0.01295461],
        [-0.03304758, -0.03274642,  0.06383443, ...,  0.00582008,
         -0.07441653, -0.05390349],
        [-0.00576582,  0.02308074,  0.02078492, ..., -0.0436268 ,
          0.00511709,  0.00323573],
        ...,
        [-0.00576636, -0.06797799, -0.08139122, ..., -0.06388375,
         -0.10037666, -0.04421329],
        [-0.06411765,  0.04515857,  0.07210004, ...,  0.03873212,
         -0.07372958, -0.06378485],
        [-0.03375746, -0.10732839, -0.11410367, ..., -0.11063471,
         -0.06417878, -0.08202204]], shape=(10000, 128), dtype=float32),
 array([[ 0.25245252, -0.11204396,  0.06159283, ..., -0.06670445,
         -0.01345666, -0.1588337 ],
        [ 0.29344326, -0.20946579,  0.09030144, ..., -0.10576804,
         -0.01049783, -0.03027877],
        [-0.18888508,  0.12661113, -0.1153727 , ..., -0.01515283,
         -0.21003337, -0.1878425 ],
        ...,
        [ 0.2685575

In [7]:
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i-3, '?') for i in encoded_review])
def preprocess_text(text):
    words=text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

In [8]:
def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input)
    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    return sentiment, prediction[0][0]

In [12]:
example_review = "This movie was fantastic! The acting was great and the plot was engaging. I really enjoyed it."
sentiment, confidence = predict_sentiment(example_review)
print(f"Review: {example_review}")
print(f"Predicted Sentiment: {sentiment} (Confidence: {confidence:.4f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Review: This movie was fantastic! The acting was great and the plot was engaging. I really enjoyed it.
Predicted Sentiment: Positive (Confidence: 0.9476)
